In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
# ============================================================
# PART 1 — Load and inspect the VIM-2 supplementary workbook
# ============================================================

import os
import pandas as pd

# Find the uploaded Excel file
excel_files = [
    file for file in os.listdir()
    if file.lower().endswith(".xlsx")
]

if not excel_files:
    raise FileNotFoundError(
        "No Excel (.xlsx) file was found in the current session."
    )

file_name = excel_files[0]

print(f"Selected workbook: {file_name}")

# Inspect available worksheets
xls = pd.ExcelFile(file_name)

print("\nAvailable worksheets:")
for i, sheet in enumerate(xls.sheet_names, start=1):
    print(f"{i}. {sheet}")

# Load the first worksheet without assuming a header row
raw_df = pd.read_excel(
    file_name,
    sheet_name=0,
    header=None
)

print("\nPreview of the first 12 raw rows:")
display(raw_df.head(12))

In [ ]:
# ============================================================
# PART 2 — Load the VIM-2 fitness score dataset
# ============================================================

# Load Supplementary File 2A without assuming a header row
raw_vim = pd.read_excel(
    file_name,
    sheet_name="SF2A Fitness Scores",
    header=None
)

# Display the first 15 rows so we can identify the true header
print("First 15 rows of the VIM-2 fitness sheet:")
display(raw_vim.head(15))

In [ ]:
# ============================================================
# PART 3 — Load Supplementary File 2A with the correct header
# ============================================================

# Row 1 contains the actual column names.
vim2 = pd.read_excel(
    file_name,
    sheet_name="SF2A Fitness Scores",
    header=1
)

# Remove completely empty rows and columns
vim2 = vim2.dropna(axis=0, how="all")
vim2 = vim2.dropna(axis=1, how="all")

# Clean column names
vim2.columns = (
    vim2.columns
    .astype(str)
    .str.strip()
    .str.replace("\n", " ", regex=False)
)

# Display the basic dataset structure
print("Dataset shape:")
print(vim2.shape)

print("\nColumn names:")
for i, col in enumerate(vim2.columns, start=1):
    print(f"{i}. {col}")

print("\nFirst 5 rows:")
display(vim2.head())

In [ ]:
# ============================================================
# PART 4 — Fitness Conditions and Missing-Value Assessment
# ============================================================

# Identify fitness-score columns.
# Standard deviation (SD) columns are intentionally excluded.

fitness_cols = [
    col for col in vim2.columns
    if "ug/mL" in col and not col.endswith("_SD")
]

# Identify corresponding standard-deviation columns.
sd_cols = [
    col for col in vim2.columns
    if col.endswith("_SD")
]

print("=" * 60)
print("FITNESS CONDITIONS")
print("=" * 60)

print(f"Number of fitness conditions: {len(fitness_cols)}\n")

for i, col in enumerate(fitness_cols, start=1):
    print(f"{i}. {col}")


print("\n" + "=" * 60)
print("STANDARD DEVIATION CONDITIONS")
print("=" * 60)

print(f"Number of SD columns: {len(sd_cols)}\n")

for i, col in enumerate(sd_cols, start=1):
    print(f"{i}. {col}")


# ------------------------------------------------------------
# Assess missing values in fitness measurements
# ------------------------------------------------------------

fitness_missing = pd.DataFrame({
    "Condition": fitness_cols,
    "Non_Missing": [
        vim2[col].notna().sum()
        for col in fitness_cols
    ],
    "Missing": [
        vim2[col].isna().sum()
        for col in fitness_cols
    ]
})

fitness_missing["Missing_Percent"] = (
    fitness_missing["Missing"]
    / len(vim2)
    * 100
).round(2)


print("\n" + "=" * 60)
print("FITNESS DATA COMPLETENESS")
print("=" * 60)

display(fitness_missing)

In [ ]:
# ============================================================
# PART 5 — Mutation Composition and Coverage
# ============================================================

# ------------------------------------------------------------
# 1. Count mutation identity categories
# ------------------------------------------------------------

print("=" * 60)
print("IDENTITY CATEGORY COUNTS")
print("=" * 60)

identity_counts = (
    vim2["identity"]
    .value_counts(dropna=False)
    .rename("Count")
    .to_frame()
)

identity_counts["Percentage"] = (
    identity_counts["Count"]
    / len(vim2)
    * 100
).round(2)

display(identity_counts)


# ------------------------------------------------------------
# 2. Count wild-type residues
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("WILD-TYPE RESIDUE COUNTS")
print("=" * 60)

wt_counts = (
    vim2["wt residue"]
    .value_counts(dropna=False)
    .rename("Count")
    .to_frame()
)

display(wt_counts)


# ------------------------------------------------------------
# 3. Count variant residues
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("VARIANT RESIDUE COUNTS")
print("=" * 60)

variant_counts = (
    vim2["variant residue"]
    .value_counts(dropna=False)
    .rename("Count")
    .to_frame()
)

display(variant_counts)


# ------------------------------------------------------------
# 4. Identify stop-codon variants
# ------------------------------------------------------------

stop_mask = (
    vim2["variant residue"]
    .astype(str)
    .str.strip()
    .eq("*")
)

stop_variants = vim2[stop_mask]

print("\n" + "=" * 60)
print("STOP-CODON VARIANTS")
print("=" * 60)

print(f"Number of stop-codon variants: {len(stop_variants)}")

if len(stop_variants) > 0:
    display(
        stop_variants[
            [
                "index",
                "wt residue",
                "position",
                "variant residue",
                "identity"
            ]
        ].head(20)
    )


# ------------------------------------------------------------
# 5. Sequence-position coverage
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SEQUENCE POSITION COVERAGE")
print("=" * 60)

print(
    f"Number of unique positions: "
    f"{vim2['position'].nunique()}"
)

print(
    f"Minimum position: "
    f"{vim2['position'].min()}"
)

print(
    f"Maximum position: "
    f"{vim2['position'].max()}"
)


# ------------------------------------------------------------
# 6. Check for duplicate amino-acid substitutions
# ------------------------------------------------------------

mutation_key = [
    "wt residue",
    "position",
    "variant residue"
]

duplicate_mask = vim2.duplicated(
    subset=mutation_key,
    keep=False
)

print("\n" + "=" * 60)
print("DUPLICATE MUTATION CHECK")
print("=" * 60)

print(
    f"Rows involved in duplicated substitutions: "
    f"{duplicate_mask.sum()}"
)

if duplicate_mask.sum() > 0:
    display(
        vim2.loc[
            duplicate_mask,
            mutation_key
        ].sort_values(mutation_key).head(30)
    )

In [ ]:
# ============================================================
# PART 5.1 — Standardize the sequence-position column
# ============================================================

# Convert position values to numeric.
# Invalid or non-numeric entries will become NaN.

vim2["position_numeric"] = pd.to_numeric(
    vim2["position"],
    errors="coerce"
)

print("=" * 60)
print("POSITION DATA TYPE CHECK")
print("=" * 60)

print("Original dtype:")
print(vim2["position"].dtype)

print("\nNumeric dtype:")
print(vim2["position_numeric"].dtype)

print("\nMissing values after numeric conversion:")
print(vim2["position_numeric"].isna().sum())

print("\nNumber of unique numeric positions:")
print(vim2["position_numeric"].nunique())

print("\nMinimum position:")
print(vim2["position_numeric"].min())

print("\nMaximum position:")
print(vim2["position_numeric"].max())

In [ ]:
# ============================================================
# PART 5.2 — Variant Identity and Stop-Codon Analysis
# ============================================================

# ------------------------------------------------------------
# 1. Count identity categories
# ------------------------------------------------------------

print("=" * 60)
print("IDENTITY CATEGORY COUNTS")
print("=" * 60)

identity_counts = (
    vim2["identity"]
    .value_counts(dropna=False)
    .rename("Count")
    .to_frame()
)

identity_counts["Percentage"] = (
    identity_counts["Count"] / len(vim2) * 100
).round(2)

display(identity_counts)


# ------------------------------------------------------------
# 2. Identify stop-codon variants
# ------------------------------------------------------------

stop_mask = (
    vim2["variant residue"]
    .astype(str)
    .str.strip()
    .eq("*")
)

stop_variants = vim2[stop_mask]

print("\n" + "=" * 60)
print("STOP-CODON VARIANTS")
print("=" * 60)

print(f"Number of stop-codon variants: {len(stop_variants)}")

if len(stop_variants) > 0:
    display(
        stop_variants[
            [
                "index",
                "wt residue",
                "position",
                "variant residue",
                "identity"
            ]
        ].head(20)
    )


# ------------------------------------------------------------
# 3. Check for duplicate amino-acid substitutions
# ------------------------------------------------------------

mutation_key = [
    "wt residue",
    "position_numeric",
    "variant residue"
]

duplicate_mask = vim2.duplicated(
    subset=mutation_key,
    keep=False
)

print("\n" + "=" * 60)
print("DUPLICATE MUTATION CHECK")
print("=" * 60)

print(
    "Rows involved in duplicated amino-acid substitutions:",
    duplicate_mask.sum()
)

if duplicate_mask.sum() > 0:
    display(
        vim2.loc[
            duplicate_mask,
            mutation_key
        ].sort_values(mutation_key).head(30)
    )

In [ ]:
# ============================================================
# PART 5.3 — Identify the Missense Mutation Dataset
# ============================================================

# Inspect all variant residue symbols
# to determine which substitutions are standard amino-acid changes.

print("=" * 60)
print("VARIANT RESIDUE COUNTS")
print("=" * 60)

variant_residue_counts = (
    vim2["variant residue"]
    .value_counts(dropna=False)
    .rename("Count")
    .to_frame()
)

display(variant_residue_counts)


# ------------------------------------------------------------
# Define standard amino-acid residues
# ------------------------------------------------------------

standard_amino_acids = set(
    list("ACDEFGHIKLMNPQRSTVWY")
)


# ------------------------------------------------------------
# Identify standard missense substitutions
# ------------------------------------------------------------

missense_mask = (
    vim2["identity"].eq("var")
    &
    vim2["variant residue"]
    .isin(standard_amino_acids)
)

missense_df = vim2[missense_mask].copy()


# ------------------------------------------------------------
# Summarize the resulting dataset
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MISSENSE MUTATION DATASET")
print("=" * 60)

print(f"Number of missense variants: {len(missense_df)}")

print(
    f"Number of unique positions represented: "
    f"{missense_df['position_numeric'].nunique()}"
)

print(
    f"Number of unique variant residues: "
    f"{missense_df['variant residue'].nunique()}"
)


# ------------------------------------------------------------
# Show a few examples
# ------------------------------------------------------------

print("\nExample missense variants:")

display(
    missense_df[
        [
            "index",
            "wt residue",
            "position_numeric",
            "variant residue",
            "identity"
        ]
    ].head(20)
)

In [ ]:
# ============================================================
# PART 6 — Mutation Coverage Across Sequence Positions
# ============================================================

# Count the number of missense variants observed at each position.

position_coverage = (
    missense_df
    .groupby("position_numeric")
    .size()
    .rename("Missense_Count")
    .to_frame()
)


# ------------------------------------------------------------
# Expected maximum number of missense substitutions
# ------------------------------------------------------------
#
# A standard amino-acid position can theoretically have
# 19 alternative amino acids (excluding the wild-type residue).
#
# We therefore compare the observed number of substitutions
# against the theoretical maximum of 19.

position_coverage["Coverage_Percent"] = (
    position_coverage["Missense_Count"] / 19 * 100
).round(2)


# ------------------------------------------------------------
# Summary statistics
# ------------------------------------------------------------

print("=" * 60)
print("MISSENSE MUTATION COVERAGE")
print("=" * 60)

print(
    f"Number of positions: "
    f"{len(position_coverage)}"
)

print(
    f"Minimum substitutions at a position: "
    f"{position_coverage['Missense_Count'].min()}"
)

print(
    f"Maximum substitutions at a position: "
    f"{position_coverage['Missense_Count'].max()}"
)

print(
    f"Mean substitutions per position: "
    f"{position_coverage['Missense_Count'].mean():.2f}"
)


# ------------------------------------------------------------
# Show the distribution of coverage
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SUBSTITUTION COUNT DISTRIBUTION")
print("=" * 60)

display(
    position_coverage["Missense_Count"]
    .value_counts()
    .sort_index()
    .rename("Number_of_Positions")
    .to_frame()
)


# ------------------------------------------------------------
# Show positions with incomplete coverage
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("POSITIONS WITH INCOMPLETE COVERAGE")
print("=" * 60)

incomplete_positions = position_coverage[
    position_coverage["Missense_Count"] < 19
].sort_values("Missense_Count")

print(
    f"Number of positions with fewer than 19 substitutions: "
    f"{len(incomplete_positions)}"
)

display(incomplete_positions.head(30))

In [ ]:
# ============================================================
# PART 7 — Correlation Between Fitness Conditions
# ============================================================

# Calculate pairwise Pearson correlations between the nine
# experimental fitness conditions using missense variants only.

fitness_corr = missense_df[fitness_cols].corr(
    method="pearson"
)

print("=" * 60)
print("FITNESS CONDITION CORRELATION MATRIX")
print("=" * 60)

display(
    fitness_corr.round(3)
)


# ------------------------------------------------------------
# Identify the strongest and weakest condition correlations
# ------------------------------------------------------------

corr_pairs = []

for i in range(len(fitness_cols)):
    for j in range(i + 1, len(fitness_cols)):

        corr_pairs.append({
            "Condition_1": fitness_cols[i],
            "Condition_2": fitness_cols[j],
            "Pearson_r": fitness_corr.iloc[i, j]
        })

corr_pairs_df = (
    pd.DataFrame(corr_pairs)
    .sort_values(
        "Pearson_r",
        ascending=False
    )
)

print("\n" + "=" * 60)
print("STRONGEST CONDITION CORRELATIONS")
print("=" * 60)

display(
    corr_pairs_df.head(10).round(3)
)


print("\n" + "=" * 60)
print("WEAKEST CONDITION CORRELATIONS")
print("=" * 60)

display(
    corr_pairs_df.tail(10).sort_values(
        "Pearson_r"
    ).round(3)
)

In [ ]:
# ============================================================
# PART 8 — Fitness Distribution and Variability
# ============================================================

# Calculate descriptive statistics for each fitness condition.

fitness_summary = pd.DataFrame({
    "Condition": fitness_cols,
    "N": [
        missense_df[col].notna().sum()
        for col in fitness_cols
    ],
    "Mean": [
        missense_df[col].mean()
        for col in fitness_cols
    ],
    "SD": [
        missense_df[col].std()
        for col in fitness_cols
    ],
    "Min": [
        missense_df[col].min()
        for col in fitness_cols
    ],
    "Median": [
        missense_df[col].median()
        for col in fitness_cols
    ],
    "Max": [
        missense_df[col].max()
        for col in fitness_cols
    ],
    "Unique": [
        missense_df[col].nunique()
        for col in fitness_cols
    ]
})

# Coefficient of variation is useful for comparing
# relative variability across conditions.
fitness_summary["CV"] = (
    fitness_summary["SD"].abs()
    / fitness_summary["Mean"].abs()
)

print("=" * 60)
print("FITNESS DISTRIBUTION SUMMARY")
print("=" * 60)

display(
    fitness_summary.round(4)
)

In [ ]:
# ============================================================
# PART 9 — BUILD A CLEAN MUTATION FEATURE TABLE
# ============================================================
#
# This section:
# 1. Keeps only missense mutations
# 2. Cleans invalid sequence positions
# 3. Creates standardized mutation labels
# 4. Adds physicochemical mutation features
# 5. Adds BLOSUM62 evolutionary scores
# 6. Performs quality-control checks
#
# The final output is a clean mutation-level feature table
# ready for downstream machine-learning analysis.
# ============================================================


# ------------------------------------------------------------
# 1. Start from the missense mutation dataset
# ------------------------------------------------------------

features_df = missense_df.copy()


# ------------------------------------------------------------
# 2. Rename columns for clarity
# ------------------------------------------------------------

features_df = features_df.rename(
    columns={
        "wt residue": "WT_AA",
        "position_numeric": "Position",
        "variant residue": "Mutant_AA"
    }
)


# ------------------------------------------------------------
# 3. Keep only valid standard amino-acid substitutions
# ------------------------------------------------------------

standard_amino_acids = [
    "A", "C", "D", "E", "F",
    "G", "H", "I", "K", "L",
    "M", "N", "P", "Q", "R",
    "S", "T", "V", "W", "Y"
]

features_df = features_df[
    features_df["WT_AA"].isin(standard_amino_acids)
    &
    features_df["Mutant_AA"].isin(standard_amino_acids)
].copy()


# ------------------------------------------------------------
# 4. Clean sequence positions
# ------------------------------------------------------------

features_df["Position"] = pd.to_numeric(
    features_df["Position"],
    errors="coerce"
)

invalid_positions = features_df["Position"].isna().sum()

print("=" * 60)
print("SEQUENCE POSITION CLEANING")
print("=" * 60)

print(
    f"Invalid/missing positions before filtering: "
    f"{invalid_positions}"
)


# Remove rows without valid sequence positions.
features_df = features_df.dropna(
    subset=["Position"]
).copy()


# Convert valid positions to integers.
features_df["Position"] = (
    features_df["Position"]
    .astype(int)
)


print(
    f"Rows remaining after filtering: "
    f"{len(features_df)}"
)

print(
    f"Unique positions remaining: "
    f"{features_df['Position'].nunique()}"
)


# ------------------------------------------------------------
# 5. Create standardized mutation labels
# ------------------------------------------------------------

features_df["Mutation"] = (
    features_df["WT_AA"]
    + features_df["Position"].astype(str)
    + features_df["Mutant_AA"]
)


print("\nExample mutation labels:")

display(
    features_df[
        [
            "Mutation",
            "WT_AA",
            "Position",
            "Mutant_AA"
        ]
    ].head(10)
)


# ============================================================
# 6. DEFINE AMINO-ACID PHYSICOCHEMICAL PROPERTIES
# ============================================================

# Kyte-Doolittle hydrophobicity scale.
hydrophobicity = {
    "A": 1.8,
    "C": 2.5,
    "D": -3.5,
    "E": -3.5,
    "F": 2.8,
    "G": -0.4,
    "H": -3.2,
    "I": 4.5,
    "K": -3.9,
    "L": 3.8,
    "M": 1.9,
    "N": -3.5,
    "P": -1.6,
    "Q": -3.5,
    "R": -4.5,
    "S": -0.8,
    "T": -0.7,
    "V": 4.2,
    "W": -0.9,
    "Y": -1.3
}


# Approximate molecular weights in Daltons.
molecular_weight = {
    "A": 89.09,
    "C": 121.15,
    "D": 133.10,
    "E": 147.13,
    "F": 165.19,
    "G": 75.07,
    "H": 155.16,
    "I": 131.17,
    "K": 146.19,
    "L": 131.17,
    "M": 149.21,
    "N": 132.12,
    "P": 115.13,
    "Q": 146.15,
    "R": 174.20,
    "S": 105.09,
    "T": 119.12,
    "V": 117.15,
    "W": 204.23,
    "Y": 181.19
}


# Simplified amino-acid charge at physiological conditions.
charge = {
    "A": 0,
    "C": 0,
    "D": -1,
    "E": -1,
    "F": 0,
    "G": 0,
    "H": 0,
    "I": 0,
    "K": 1,
    "L": 0,
    "M": 0,
    "N": 0,
    "P": 0,
    "Q": 0,
    "R": 1,
    "S": 0,
    "T": 0,
    "V": 0,
    "W": 0,
    "Y": 0
}


# Simplified polarity classification:
# 1 = polar
# 0 = non-polar
polarity = {
    "A": 0,
    "C": 1,
    "D": 1,
    "E": 1,
    "F": 0,
    "G": 0,
    "H": 1,
    "I": 0,
    "K": 1,
    "L": 0,
    "M": 0,
    "N": 1,
    "P": 0,
    "Q": 1,
    "R": 1,
    "S": 1,
    "T": 1,
    "V": 0,
    "W": 0,
    "Y": 1
}


# ============================================================
# 7. ADD PHYSICOCHEMICAL FEATURES
# ============================================================

features_df["WT_Hydrophobicity"] = (
    features_df["WT_AA"]
    .map(hydrophobicity)
)

features_df["Mutant_Hydrophobicity"] = (
    features_df["Mutant_AA"]
    .map(hydrophobicity)
)

features_df["Hydrophobicity_Change"] = (
    features_df["Mutant_Hydrophobicity"]
    -
    features_df["WT_Hydrophobicity"]
)


features_df["WT_Weight"] = (
    features_df["WT_AA"]
    .map(molecular_weight)
)

features_df["Mutant_Weight"] = (
    features_df["Mutant_AA"]
    .map(molecular_weight)
)

features_df["Weight_Change"] = (
    features_df["Mutant_Weight"]
    -
    features_df["WT_Weight"]
)


features_df["WT_Charge"] = (
    features_df["WT_AA"]
    .map(charge)
)

features_df["Mutant_Charge"] = (
    features_df["Mutant_AA"]
    .map(charge)
)

features_df["Charge_Change"] = (
    features_df["Mutant_Charge"]
    -
    features_df["WT_Charge"]
)


features_df["WT_Polarity"] = (
    features_df["WT_AA"]
    .map(polarity)
)

features_df["Mutant_Polarity"] = (
    features_df["Mutant_AA"]
    .map(polarity)
)

features_df["Polarity_Change"] = (
    features_df["Mutant_Polarity"]
    -
    features_df["WT_Polarity"]
)


# ============================================================
# 8. ADD BLOSUM62 EVOLUTIONARY FEATURE
# ============================================================

# Install Biopython if it is not already available.
!pip -q install biopython

from Bio.Align import substitution_matrices


# Load the BLOSUM62 substitution matrix.
blosum62 = substitution_matrices.load("BLOSUM62")


# Calculate one BLOSUM62 score for each mutation.
features_df["BLOSUM62"] = features_df.apply(
    lambda row: blosum62[
        row["WT_AA"],
        row["Mutant_AA"]
    ],
    axis=1
)


# ============================================================
# 9. FINAL QUALITY-CONTROL CHECKS
# ============================================================

print("\n" + "=" * 60)
print("FINAL PART 9 QUALITY CONTROL")
print("=" * 60)


# Check expected feature columns.
expected_feature_columns = [
    "Mutation",
    "WT_AA",
    "Position",
    "Mutant_AA",
    "WT_Hydrophobicity",
    "Mutant_Hydrophobicity",
    "Hydrophobicity_Change",
    "WT_Weight",
    "Mutant_Weight",
    "Weight_Change",
    "WT_Charge",
    "Mutant_Charge",
    "Charge_Change",
    "WT_Polarity",
    "Mutant_Polarity",
    "Polarity_Change",
    "BLOSUM62"
]


missing_feature_columns = [
    column
    for column in expected_feature_columns
    if column not in features_df.columns
]


if len(missing_feature_columns) == 0:
    print("All expected feature columns are present.")
else:
    print(
        "Missing feature columns:",
        missing_feature_columns
    )


# Check for missing values in the final feature set.
missing_values = (
    features_df[
        expected_feature_columns
    ]
    .isna()
    .sum()
)


print("\nMissing values in final feature table:")

display(
    missing_values[
        missing_values > 0
    ]
)


if missing_values.sum() == 0:
    print(
        "No missing values detected "
        "in the final feature table."
    )


# Check mutation duplicates.
duplicate_mutations = (
    features_df["Mutation"]
    .duplicated()
    .sum()
)


print(
    f"\nDuplicate mutation labels: "
    f"{duplicate_mutations}"
)


# Check position range.
print(
    f"\nMinimum sequence position: "
    f"{features_df['Position'].min()}"
)

print(
    f"Maximum sequence position: "
    f"{features_df['Position'].max()}"
)


# ============================================================
# 10. FINAL DATASET SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("PART 9 COMPLETE — FINAL FEATURE TABLE")
print("=" * 60)

print(
    f"Final number of mutations: "
    f"{len(features_df)}"
)

print(
    f"Number of unique positions: "
    f"{features_df['Position'].nunique()}"
)

print(
    f"Number of unique mutations: "
    f"{features_df['Mutation'].nunique()}"
)

print(
    f"Number of final features: "
    f"{len(expected_feature_columns) - 4}"
)


print("\nFinal feature table preview:")

display(
    features_df[
        expected_feature_columns
    ].head(10)
)

In [ ]:
# ============================================================
# SAVE FINAL PART 9 FEATURE DATASET
# ============================================================
#
# This file contains the cleaned VIM-2 missense mutation
# dataset together with all engineered mutation features.
#
# It can be loaded directly in future sessions without
# rerunning the previous data-cleaning steps.
# ============================================================

from google.colab import files


# ------------------------------------------------------------
# Define the output filename
# ------------------------------------------------------------

output_filename = (
    "VIM2_Part9_Mutation_Features.csv"
)


# ------------------------------------------------------------
# Save the complete feature table
# ------------------------------------------------------------

features_df.to_csv(
    output_filename,
    index=False
)


# ------------------------------------------------------------
# Confirm that the file was saved
# ------------------------------------------------------------

print("=" * 60)
print("DATASET SAVED SUCCESSFULLY")
print("=" * 60)

print(f"Filename: {output_filename}")
print(f"Rows: {len(features_df)}")
print(f"Columns: {len(features_df.columns)}")


# ------------------------------------------------------------
# Download the dataset to your computer
# ------------------------------------------------------------

files.download(output_filename)

In [ ]:
# ============================================================
# PART 10A — CREATE MACHINE LEARNING TARGET VARIABLES
# ============================================================
#
# This section creates biologically meaningful prediction
# targets from the VIM-2 fitness measurements.
#
# We do not train any model yet.
# First, we define and inspect the prediction targets.
# ============================================================


import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Define the nine original fitness conditions
# ------------------------------------------------------------

fitness_columns = [
    "128ug/mL_AMP_25C",
    "16ug/mL_AMP_25C",
    "2ug/mL_AMP_25C",
    "128ug/mL_AMP_37C",
    "16ug/mL_AMP_37C",
    "2ug/mL_AMP_37C",
    "4ug/mL_CTX_37C",
    "0.5ug/mL_CTX_37C",
    "0.031ug/mL_MEM_37C"
]


# ------------------------------------------------------------
# 2. Verify that all original fitness columns are available
# ------------------------------------------------------------

missing_fitness_columns = [
    column
    for column in fitness_columns
    if column not in features_df.columns
]


print("=" * 60)
print("FITNESS TARGET COLUMN CHECK")
print("=" * 60)


if len(missing_fitness_columns) == 0:
    print(
        "All 9 fitness columns are available."
    )
else:
    print(
        "Missing fitness columns:"
    )
    print(
        missing_fitness_columns
    )


# ------------------------------------------------------------
# 3. Define antibiotic-specific condition groups
# ------------------------------------------------------------

amp_columns = [
    "128ug/mL_AMP_25C",
    "16ug/mL_AMP_25C",
    "2ug/mL_AMP_25C",
    "128ug/mL_AMP_37C",
    "16ug/mL_AMP_37C",
    "2ug/mL_AMP_37C"
]


ctx_columns = [
    "4ug/mL_CTX_37C",
    "0.5ug/mL_CTX_37C"
]


mem_columns = [
    "0.031ug/mL_MEM_37C"
]


# ------------------------------------------------------------
# 4. Create the overall mean fitness target
# ------------------------------------------------------------

features_df["Mean_Fitness"] = (
    features_df[fitness_columns]
    .mean(axis=1)
)


# ------------------------------------------------------------
# 5. Create antibiotic-specific fitness targets
# ------------------------------------------------------------

features_df["Fitness_AMP"] = (
    features_df[amp_columns]
    .mean(axis=1)
)


features_df["Fitness_CTX"] = (
    features_df[ctx_columns]
    .mean(axis=1)
)


features_df["Fitness_MEM"] = (
    features_df[mem_columns]
    .mean(axis=1)
)


# ------------------------------------------------------------
# 6. Measure fitness variability across conditions
# ------------------------------------------------------------
#
# Low variability:
# Mutation behaves similarly across environments.
#
# High variability:
# Mutation is strongly environment-dependent.
# ------------------------------------------------------------

features_df["Fitness_Std"] = (
    features_df[fitness_columns]
    .std(axis=1)
)


# ------------------------------------------------------------
# 7. Create a global beneficial classification target
# ------------------------------------------------------------
#
# Mean_Fitness > 0:
# Overall beneficial relative to wild type.
#
# Mean_Fitness <= 0:
# Neutral or deleterious overall.
# ------------------------------------------------------------

features_df["Global_Beneficial"] = (
    features_df["Mean_Fitness"] > 0
).astype(int)


# ------------------------------------------------------------
# 8. Create a globally positive classification target
# ------------------------------------------------------------
#
# A mutation is globally positive only if its fitness is
# greater than zero in ALL available conditions.
# ------------------------------------------------------------

features_df["Globally_Positive"] = (
    (
        features_df[fitness_columns] > 0
    )
    .all(axis=1)
).astype(int)


# ------------------------------------------------------------
# 9. Count the number of positive environments
# ------------------------------------------------------------
#
# This gives a more detailed measure than a simple
# yes/no beneficial classification.
# ------------------------------------------------------------

features_df["Positive_Condition_Count"] = (
    features_df[fitness_columns]
    .gt(0)
    .sum(axis=1)
)


# ------------------------------------------------------------
# 10. Create an environment-dependent mutation indicator
# ------------------------------------------------------------
#
# We temporarily define environment-dependent mutations as
# those with relatively high fitness variability.
#
# The threshold is the 75th percentile of Fitness_Std.
# ------------------------------------------------------------

environment_dependence_threshold = (
    features_df["Fitness_Std"]
    .quantile(0.75)
)


features_df["Environment_Dependent"] = (
    features_df["Fitness_Std"]
    >
    environment_dependence_threshold
).astype(int)


# ============================================================
# 11. TARGET QUALITY CONTROL
# ============================================================

target_columns = [
    "Mean_Fitness",
    "Fitness_AMP",
    "Fitness_CTX",
    "Fitness_MEM",
    "Fitness_Std",
    "Global_Beneficial",
    "Globally_Positive",
    "Positive_Condition_Count",
    "Environment_Dependent"
]


print("\n" + "=" * 60)
print("TARGET VARIABLE MISSING VALUE CHECK")
print("=" * 60)

target_missing = (
    features_df[target_columns]
    .isna()
    .sum()
)

display(target_missing)


# ============================================================
# 12. TARGET DISTRIBUTION SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("CONTINUOUS TARGET SUMMARY")
print("=" * 60)

continuous_targets = [
    "Mean_Fitness",
    "Fitness_AMP",
    "Fitness_CTX",
    "Fitness_MEM",
    "Fitness_Std",
    "Positive_Condition_Count"
]

display(
    features_df[
        continuous_targets
    ].describe().T
)


# ============================================================
# 13. CLASSIFICATION TARGET COUNTS
# ============================================================

print("\n" + "=" * 60)
print("GLOBAL BENEFICIAL DISTRIBUTION")
print("=" * 60)

display(
    features_df[
        "Global_Beneficial"
    ]
    .value_counts()
    .sort_index()
)


print("\n" + "=" * 60)
print("GLOBALLY POSITIVE DISTRIBUTION")
print("=" * 60)

display(
    features_df[
        "Globally_Positive"
    ]
    .value_counts()
    .sort_index()
)


print("\n" + "=" * 60)
print("ENVIRONMENT-DEPENDENT DISTRIBUTION")
print("=" * 60)

display(
    features_df[
        "Environment_Dependent"
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# 14. FINAL TARGET TABLE PREVIEW
# ============================================================

print("\n" + "=" * 60)
print("PART 10A COMPLETE")
print("=" * 60)

print(
    f"Total mutations: "
    f"{len(features_df)}"
)

print(
    f"Environment-dependence threshold: "
    f"{environment_dependence_threshold:.4f}"
)


print("\nTarget table preview:")

display(
    features_df[
        [
            "Mutation",
            "Mean_Fitness",
            "Fitness_AMP",
            "Fitness_CTX",
            "Fitness_MEM",
            "Fitness_Std",
            "Global_Beneficial",
            "Globally_Positive",
            "Positive_Condition_Count",
            "Environment_Dependent"
        ]
    ].head(10)
)

In [ ]:
# ============================================================
# BROADLY POSITIVE MUTATIONS
# ============================================================

# A mutation is considered broadly positive if it has
# positive fitness in at least 7 out of the 9 conditions.

features_df["Broadly_Positive"] = (
    features_df["Positive_Condition_Count"] >= 7
).astype(int)


# ============================================================
# BROADLY POSITIVE DISTRIBUTION
# ============================================================

print("\n" + "=" * 60)
print("BROADLY POSITIVE DISTRIBUTION")
print("=" * 60)

broadly_counts = (
    features_df["Broadly_Positive"]
    .value_counts()
    .sort_index()
)

broadly_percentages = (
    features_df["Broadly_Positive"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

broadly_summary = pd.DataFrame({
    "Count": broadly_counts,
    "Percentage": broadly_percentages.round(2)
})

display(broadly_summary)


# ============================================================
# POSITIVE CONDITION COUNT DISTRIBUTION
# ============================================================

print("\n" + "=" * 60)
print("POSITIVE CONDITION COUNT DISTRIBUTION")
print("=" * 60)

positive_count_distribution = (
    features_df["Positive_Condition_Count"]
    .value_counts()
    .sort_index()
)

display(positive_count_distribution)


# ============================================================
# CHECK RELATIONSHIP BETWEEN BROADLY POSITIVE
# AND GLOBALLY POSITIVE
# ============================================================

print("\n" + "=" * 60)
print("BROADLY POSITIVE VS GLOBALLY POSITIVE")
print("=" * 60)

relationship_table = pd.crosstab(
    features_df["Broadly_Positive"],
    features_df["Globally_Positive"],
    margins=True
)

display(relationship_table)


# ============================================================
# EXAMPLE BROADLY POSITIVE MUTATIONS
# ============================================================

print("\n" + "=" * 60)
print("EXAMPLE BROADLY POSITIVE MUTATIONS")
print("=" * 60)

broadly_examples = (
    features_df[
        features_df["Broadly_Positive"] == 1
    ]
    [
        [
            "Mutation",
            "Mean_Fitness",
            "Positive_Condition_Count",
            "Global_Beneficial",
            "Globally_Positive",
            "Broadly_Positive",
            "Environment_Dependent"
        ]
    ]
    .sort_values(
        by=[
            "Positive_Condition_Count",
            "Mean_Fitness"
        ],
        ascending=[False, False]
    )
)

display(broadly_examples.head(20))


# ============================================================
# PART 10A — UPDATED TARGET SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("PART 10A — UPDATED TARGET SUMMARY")
print("=" * 60)

print(f"Total mutations: {len(features_df)}")

print(
    f"Global beneficial mutations: "
    f"{features_df['Global_Beneficial'].sum()} "
    f"({features_df['Global_Beneficial'].mean() * 100:.2f}%)"
)

print(
    f"Broadly positive mutations (>= 7/9): "
    f"{features_df['Broadly_Positive'].sum()} "
    f"({features_df['Broadly_Positive'].mean() * 100:.2f}%)"
)

print(
    f"Globally positive mutations (9/9): "
    f"{features_df['Globally_Positive'].sum()} "
    f"({features_df['Globally_Positive'].mean() * 100:.2f}%)"
)

print(
    f"Environment-dependent mutations: "
    f"{features_df['Environment_Dependent'].sum()} "
    f"({features_df['Environment_Dependent'].mean() * 100:.2f}%)"
)


# ============================================================
# SAVE UPDATED DATASET
# ============================================================

features_df.to_csv(
    "vim2_ml_features_part10a_updated.csv",
    index=False
)

print("\nUpdated dataset saved successfully:")
print("vim2_ml_features_part10a_updated.csv")

In [ ]:
# ============================================================
# PART 10B — TARGET VALIDATION & RELATIONSHIP ANALYSIS
# ============================================================

print("=" * 60)
print("PART 10B — TARGET VALIDATION & RELATIONSHIP ANALYSIS")
print("=" * 60)

# ------------------------------------------------------------
# 1. CHECK REQUIRED TARGET COLUMNS
# ------------------------------------------------------------

target_columns = [
    "Global_Beneficial",
    "Broadly_Positive",
    "Globally_Positive",
    "Environment_Dependent"
]

missing_targets = [
    col for col in target_columns
    if col not in features_df.columns
]

if missing_targets:
    raise ValueError(
        f"Missing target columns: {missing_targets}"
    )

print("\nAll required target columns are present.")


# ============================================================
# 2. TARGET DISTRIBUTIONS
# ============================================================

print("\n" + "=" * 60)
print("TARGET DISTRIBUTIONS")
print("=" * 60)

target_summary = []

for col in target_columns:

    count_1 = int(features_df[col].sum())
    count_0 = int((features_df[col] == 0).sum())
    total = len(features_df)

    target_summary.append({
        "Target": col,
        "Positive_Count": count_1,
        "Negative_Count": count_0,
        "Positive_Percent": round(
            count_1 / total * 100, 2
        ),
        "Negative_Percent": round(
            count_0 / total * 100, 2
        )
    })

target_summary_df = pd.DataFrame(target_summary)

display(target_summary_df)


# ============================================================
# 3. POSITIVE CONDITION COUNT DISTRIBUTION
# ============================================================

print("\n" + "=" * 60)
print("POSITIVE CONDITION COUNT DISTRIBUTION")
print("=" * 60)

positive_count_distribution = (
    features_df["Positive_Condition_Count"]
    .value_counts()
    .sort_index()
)

positive_count_percent = (
    features_df["Positive_Condition_Count"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

positive_count_summary = pd.DataFrame({
    "Count": positive_count_distribution,
    "Percentage": positive_count_percent.round(2)
})

display(positive_count_summary)


# ============================================================
# 4. LOGICAL NESTING CHECK
# ============================================================

print("\n" + "=" * 60)
print("LOGICAL NESTING CHECK")
print("=" * 60)

# Every globally positive mutation should also be broadly positive.
globally_positive_not_broadly = features_df[
    (features_df["Globally_Positive"] == 1) &
    (features_df["Broadly_Positive"] == 0)
]

print(
    "Globally positive but NOT broadly positive: "
    f"{len(globally_positive_not_broadly)}"
)

# Every broadly positive mutation should have at least 7 positive conditions.
broadly_invalid = features_df[
    (features_df["Broadly_Positive"] == 1) &
    (features_df["Positive_Condition_Count"] < 7)
]

print(
    "Broadly positive with fewer than 7 positive conditions: "
    f"{len(broadly_invalid)}"
)

# Every globally positive mutation should have 9 positive conditions.
globally_invalid = features_df[
    (features_df["Globally_Positive"] == 1) &
    (features_df["Positive_Condition_Count"] < 9)
]

print(
    "Globally positive with fewer than 9 positive conditions: "
    f"{len(globally_invalid)}"
)


# ============================================================
# 5. CROSS-TARGET RELATIONSHIP
# ============================================================

print("\n" + "=" * 60)
print("CROSS-TARGET RELATIONSHIP")
print("=" * 60)

target_relationship = pd.crosstab(
    features_df["Global_Beneficial"],
    [
        features_df["Broadly_Positive"],
        features_df["Globally_Positive"]
    ],
    margins=True
)

display(target_relationship)


# ============================================================
# 6. ENVIRONMENT-DEPENDENT VS POSITIVITY
# ============================================================

print("\n" + "=" * 60)
print("ENVIRONMENT-DEPENDENT VS POSITIVITY")
print("=" * 60)

environment_relationship = pd.crosstab(
    features_df["Environment_Dependent"],
    features_df["Broadly_Positive"],
    margins=True
)

display(environment_relationship)


# ============================================================
# 7. ENVIRONMENT-DEPENDENT MUTATIONS
#    — POSITIVE CONDITION PROFILE
# ============================================================

print("\n" + "=" * 60)
print("ENVIRONMENT-DEPENDENT MUTATIONS — POSITIVE CONDITION PROFILE")
print("=" * 60)

environment_profile = (
    features_df
    .groupby("Environment_Dependent")[
        "Positive_Condition_Count"
    ]
    .agg(
        Count="count",
        Mean="mean",
        Median="median",
        Min="min",
        Max="max"
    )
    .round(2)
)

display(environment_profile)


# ============================================================
# 8. TARGET OVERLAP COUNTS
# ============================================================

print("\n" + "=" * 60)
print("TARGET OVERLAP COUNTS")
print("=" * 60)

overlap_conditions = {
    "Global_Beneficial + Broadly_Positive":
        (
            (features_df["Global_Beneficial"] == 1) &
            (features_df["Broadly_Positive"] == 1)
        ),

    "Broadly_Positive + Globally_Positive":
        (
            (features_df["Broadly_Positive"] == 1) &
            (features_df["Globally_Positive"] == 1)
        ),

    "Global_Beneficial + Globally_Positive":
        (
            (features_df["Global_Beneficial"] == 1) &
            (features_df["Globally_Positive"] == 1)
        ),

    "Environment_Dependent + Broadly_Positive":
        (
            (features_df["Environment_Dependent"] == 1) &
            (features_df["Broadly_Positive"] == 1)
        ),

    "Environment_Dependent + Globally_Positive":
        (
            (features_df["Environment_Dependent"] == 1) &
            (features_df["Globally_Positive"] == 1)
        )
}

overlap_results = []

for name, condition in overlap_conditions.items():

    count = int(condition.sum())

    overlap_results.append({
        "Relationship": name,
        "Count": count,
        "Percentage_of_all_mutations":
            round(count / len(features_df) * 100, 2)
    })

overlap_df = pd.DataFrame(overlap_results)

display(overlap_df)


# ============================================================
# 9. MUTATIONS THAT ARE AVERAGE-BENEFICIAL
#    BUT NOT BROADLY POSITIVE
# ============================================================

print("\n" + "=" * 60)
print("GLOBAL BENEFICIAL BUT NOT BROADLY POSITIVE")
print("=" * 60)

average_only = features_df[
    (features_df["Global_Beneficial"] == 1) &
    (features_df["Broadly_Positive"] == 0)
].copy()

print(
    "Number of mutations: "
    f"{len(average_only)}"
)

display(
    average_only[
        [
            "Mutation",
            "Mean_Fitness",
            "Positive_Condition_Count",
            "Global_Beneficial",
            "Broadly_Positive",
            "Globally_Positive",
            "Environment_Dependent"
        ]
    ]
    .sort_values(
        by="Mean_Fitness",
        ascending=False
    )
    .head(20)
)


# ============================================================
# 10. BROADLY POSITIVE BUT NOT GLOBALLY POSITIVE
# ============================================================

print("\n" + "=" * 60)
print("BROADLY POSITIVE BUT NOT GLOBALLY POSITIVE")
print("=" * 60)

broadly_not_global = features_df[
    (features_df["Broadly_Positive"] == 1) &
    (features_df["Globally_Positive"] == 0)
].copy()

print(
    "Number of mutations: "
    f"{len(broadly_not_global)}"
)

display(
    broadly_not_global[
        [
            "Mutation",
            "Mean_Fitness",
            "Positive_Condition_Count",
            "Global_Beneficial",
            "Broadly_Positive",
            "Globally_Positive",
            "Environment_Dependent"
        ]
    ]
    .sort_values(
        by=[
            "Positive_Condition_Count",
            "Mean_Fitness"
        ],
        ascending=[False, False]
    )
    .head(20)
)


# ============================================================
# 11. GLOBALLY POSITIVE MUTATIONS
# ============================================================

print("\n" + "=" * 60)
print("GLOBALLY POSITIVE MUTATIONS")
print("=" * 60)

globally_positive = features_df[
    features_df["Globally_Positive"] == 1
].copy()

print(
    "Number of globally positive mutations: "
    f"{len(globally_positive)}"
)

display(
    globally_positive[
        [
            "Mutation",
            "Mean_Fitness",
            "Positive_Condition_Count",
            "Global_Beneficial",
            "Broadly_Positive",
            "Globally_Positive",
            "Environment_Dependent"
        ]
    ]
    .sort_values(
        by="Mean_Fitness",
        ascending=False
    )
    .head(20)
)


# ============================================================
# 12. FINAL PART 10B SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("PART 10B — FINAL SUMMARY")
print("=" * 60)

print(
    f"Total mutations: "
    f"{len(features_df)}"
)

for col in target_columns:

    count = int(features_df[col].sum())
    percentage = (
        features_df[col].mean() * 100
    )

    print(
        f"{col}: "
        f"{count} "
        f"({percentage:.2f}%)"
    )

print(
    "\nGlobally positive but not broadly positive: "
    f"{len(globally_positive_not_broadly)}"
)

print(
    "Broadly positive but not globally positive: "
    f"{len(broadly_not_global)}"
)

print(
    "Global beneficial but not broadly positive: "
    f"{len(average_only)}"
)

print(
    "\nPart 10B validation completed successfully."
)


# ============================================================
# SAVE PART 10B DATASET
# ============================================================

features_df.to_csv(
    "vim2_ml_features_part10b_validated.csv",
    index=False
)

print(
    "\nDataset saved successfully:"
)

print(
    "vim2_ml_features_part10b_validated.csv"
)

In [ ]:
from google.colab import files

files.download("vim2_ml_features_part10b_validated.csv")